In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader, Subset
import numpy as np
import matplotlib.pyplot as plt
import copy
import os

In [2]:
# ==========================================
# 1. Hardware Optimization & Config
# ==========================================
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

# Enable CuDNN benchmark for faster convolutions on static input sizes
if torch.cuda.is_available():
    torch.backends.cudnn.benchmark = True

EPOCHS_BASE = 200
EPOCHS_RETAIN = 200
# BATCH_SIZE = 128
# LR = 0.01
BATCH_SIZE = 1024
LR = 0.08
CHECKPOINT_INTERVAL = 40  

Using device: cuda


In [3]:
# ==========================================
# 2. Data Loading
# ==========================================
transform_train = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
])

transform_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
])

trainset = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=transform_train)
testset = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=transform_test)

testloader = DataLoader(testset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
all_indices = np.arange(len(trainset))

HTTPError: HTTP Error 403: Forbidden

In [4]:
# ==========================================
# 3. Utilities
# ==========================================
def set_seed(seed):
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    np.random.seed(seed)

def get_resnet18():
    model = torchvision.models.resnet18(weights=None)
    model.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
    model.maxpool = nn.Identity()
    model.fc = nn.Linear(model.fc.in_features, 10)
    return model.to(DEVICE)

def evaluate_model(model, dataloader, target_class=None):
    model.eval()
    overall_correct, overall_total = 0, 0
    class_correct, class_total = 0, 0
    
    with torch.no_grad():
        for inputs, targets in dataloader:
            inputs, targets = inputs.to(DEVICE), targets.to(DEVICE)
            outputs = model(inputs)
            _, predicted = outputs.max(1)
            
            # Overall accuracy calculation
            overall_total += targets.size(0)
            overall_correct += predicted.eq(targets).sum().item()
            
            # Class-specific accuracy calculation
            if target_class is not None:
                class_mask = (targets == target_class)
                class_total += class_mask.sum().item()
                class_correct += (predicted[class_mask] == targets[class_mask]).sum().item()
                
    overall_acc = 100. * overall_correct / overall_total
    
    # If a target class is requested, return both
    if target_class is not None:
        class_acc = 100. * class_correct / class_total if class_total > 0 else 0.0
        return overall_acc, class_acc
        
    return overall_acc

In [5]:
# ==========================================
# 3.5 Hardware Check
# ==========================================
print("=== Hardware Diagnostic ===")

# Check CPU Cores
print(f"Available CPU Cores: {os.cpu_count()}")

# Check GPU
if torch.cuda.is_available():
    print(f"GPU Model: {torch.cuda.get_device_name(0)}")
    
    # Check GPU Memory (VRAM)
    allocated = torch.cuda.memory_allocated(0) / 1024**3
    reserved = torch.cuda.memory_reserved(0) / 1024**3
    print(f"VRAM Allocated: {allocated:.2f} GB")
    print(f"VRAM Reserved:  {reserved:.2f} GB")
else:
    print("CUDA is NOT available! You are running on the CPU.")

=== Hardware Diagnostic ===
Available CPU Cores: 128
GPU Model: NVIDIA A100-PCIE-40GB
VRAM Allocated: 0.00 GB
VRAM Reserved:  0.00 GB


In [6]:
# ==========================================
# 4. Base Training & TracIn Influence (with VRAM tracking)
# ==========================================
set_seed(42)
base_model = get_resnet18()

# [NEW] Compile the model to bypass Python overhead (Requires PyTorch 2.0+)
# try:
#     base_model = torch.compile(base_model)
#     print("Model compiled successfully!")
# except Exception as e:
#     print("Could not compile model (likely older PyTorch version). Continuing...")

criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(base_model.parameters(), lr=LR, momentum=0.9, weight_decay=5e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS_BASE)

# [NEW] Initialize the AMP Gradient Scaler
scaler = torch.cuda.amp.GradScaler()

trainloader_base = DataLoader(trainset, batch_size=BATCH_SIZE, shuffle=True, num_workers=8, pin_memory=True, drop_last=False)
saved_checkpoints = []

print("\n--- Starting Base Model Training ---")

if torch.cuda.is_available():
    torch.cuda.reset_peak_memory_stats(DEVICE)

base_model.train()
for epoch in range(EPOCHS_BASE):
    running_loss = 0.0
    for inputs, targets in trainloader_base:
        inputs, targets = inputs.to(DEVICE), targets.to(DEVICE)
        
        optimizer.zero_grad(set_to_none=True) # slightly faster than standard zero_grad()
        
        # [NEW] Wrap the forward pass in autocast for Tensor Cores
        with torch.cuda.amp.autocast():
            loss = criterion(base_model(inputs), targets)
            
        # [NEW] Scale the loss and step
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        
        running_loss += loss.item()
    scheduler.step()
    
    if (epoch + 1) % CHECKPOINT_INTERVAL == 0:
        saved_checkpoints.append(copy.deepcopy(base_model.state_dict()))
        
    if (epoch + 1) % 20 == 0 or epoch == 0:
        if torch.cuda.is_available():
            peak_vram = torch.cuda.max_memory_allocated(DEVICE) / (1024 ** 3)
            vram_str = f"| Peak VRAM: {peak_vram:.2f} GB"
        else:
            vram_str = ""
            
        print(f"Base Epoch {epoch+1}/{EPOCHS_BASE} | Loss: {running_loss/len(trainloader_base):.4f} {vram_str}")

print("--- Computing TracIn Influence Scores ---")
eval_loader = DataLoader(trainset, batch_size=1, shuffle=False, num_workers=8, pin_memory=True)
influence_scores = {i: 0.0 for i in range(len(trainset))}

for cp_idx, state_dict in enumerate(saved_checkpoints):
    print(f"Processing Checkpoint {cp_idx + 1}/{len(saved_checkpoints)}...")
    temp_model = get_resnet18()
    temp_model.load_state_dict(state_dict)
    temp_model.eval()
    
    for i, (inputs, targets) in enumerate(eval_loader):
        inputs, targets = inputs.to(DEVICE), targets.to(DEVICE)
        temp_model.zero_grad()
        loss = criterion(temp_model(inputs), targets)
        loss.backward()
        
        grad_norm = 0.0
        for param in temp_model.fc.parameters():
            if param.grad is not None:
                grad_norm += param.grad.data.norm(2).item() ** 2
        influence_scores[i] += grad_norm

influence_ranking = sorted(influence_scores.items(), key=lambda x: x[1], reverse=True)
top_global_indices = [x[0] for x in influence_ranking]
print("Influence computation complete.\n")


--- Starting Base Model Training ---


/scratch/ss17886/conda/envs/deeplearning/lib/python3.10/site-packages/torch/nn/modules/conv.py:456: UserWarning: Applied workaround for CuDNN issue, install nvrtc.so (Triggered internally at ../aten/src/ATen/native/cudnn/Conv_v8.cpp:80.)
  return F.conv2d(input, weight, bias, self.stride,


Base Epoch 1/200 | Loss: 2.1834 | Peak VRAM: 10.84 GB


KeyboardInterrupt: 

In [ ]:
# # ==========================================
# # 4. Base Training & TracIn Influence (with VRAM tracking)
# # ==========================================
# set_seed(42)
# base_model = get_resnet18()
# criterion = nn.CrossEntropyLoss()
# optimizer = optim.SGD(base_model.parameters(), lr=LR, momentum=0.9, weight_decay=5e-4)
# scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS_BASE)

# # Optimized base trainloader
# trainloader_base = DataLoader(trainset, batch_size=BATCH_SIZE, shuffle=True, num_workers=8, pin_memory=True)
# saved_checkpoints = []

# print("\n--- Starting Base Model Training ---")

# # Reset the peak memory tracker before we start
# if torch.cuda.is_available():
#     torch.cuda.reset_peak_memory_stats(DEVICE)

# base_model.train()
# for epoch in range(EPOCHS_BASE):
#     running_loss = 0.0
#     for inputs, targets in trainloader_base:
#         inputs, targets = inputs.to(DEVICE), targets.to(DEVICE)
#         optimizer.zero_grad()
#         loss = criterion(base_model(inputs), targets)
#         loss.backward()
#         optimizer.step()
#         running_loss += loss.item()
#     scheduler.step()
    
#     if (epoch + 1) % CHECKPOINT_INTERVAL == 0:
#         saved_checkpoints.append(copy.deepcopy(base_model.state_dict()))
        
#     # Print progress AND Peak VRAM
#     if (epoch + 1) % 20 == 0 or epoch == 0:
#         if torch.cuda.is_available():
#             peak_vram = torch.cuda.max_memory_allocated(DEVICE) / (1024 ** 3) # Convert bytes to GB
#             vram_str = f"| Peak VRAM: {peak_vram:.2f} GB"
#         else:
#             vram_str = ""
            
#         print(f"Base Epoch {epoch+1}/{EPOCHS_BASE} | Loss: {running_loss/len(trainloader_base):.4f} {vram_str}")

# base_acc = evaluate_model(base_model, testloader)
# print(f"Base Model Final Accuracy: {base_acc:.2f}%\n")

# print("--- Computing TracIn Influence Scores ---")
# eval_loader = DataLoader(trainset, batch_size=1, shuffle=False, num_workers=8, pin_memory=True)
# influence_scores = {i: 0.0 for i in range(len(trainset))}

# for cp_idx, state_dict in enumerate(saved_checkpoints):
#     print(f"Processing Checkpoint {cp_idx + 1}/{len(saved_checkpoints)}...")
#     temp_model = get_resnet18()
#     temp_model.load_state_dict(state_dict)
#     temp_model.eval()
    
#     for i, (inputs, targets) in enumerate(eval_loader):
#         inputs, targets = inputs.to(DEVICE), targets.to(DEVICE)
#         temp_model.zero_grad()
#         loss = criterion(temp_model(inputs), targets)
#         loss.backward()
        
#         grad_norm = 0.0
#         for param in temp_model.fc.parameters():
#             if param.grad is not None:
#                 grad_norm += param.grad.data.norm(2).item() ** 2
#         influence_scores[i] += grad_norm

# influence_ranking = sorted(influence_scores.items(), key=lambda x: x[1], reverse=True)
# top_global_indices = [x[0] for x in influence_ranking]
# print("Influence computation complete.\n")

In [ ]:
# ==========================================
# 5. Global K-Removal Experiment (Single Seed)
# ==========================================
SEEDS = [42]  
K_VALUES = [500, 1000, 1500, 2000]

global_results = {"k_values": K_VALUES, "top_acc": {k: [] for k in K_VALUES}, "rand_acc": {k: [] for k in K_VALUES}}

print("--- Starting Global Removal Experiment ---")
for k in K_VALUES:
    print(f"\nEvaluating K = {k}")
    indices_keep_top = list(set(all_indices) - set(top_global_indices[:k]))
    subset_top = Subset(trainset, indices_keep_top)
    
    for seed in SEEDS:
        set_seed(seed)
        
        # Top-K
        print(f"  -> Training Top-{k} Removed Model...")
        loader_top = DataLoader(subset_top, batch_size=BATCH_SIZE, shuffle=True, num_workers=8, pin_memory=True)
        model_top = get_resnet18()
        optimizer_top = optim.SGD(model_top.parameters(), lr=LR, momentum=0.9, weight_decay=5e-4)
        scheduler_top = optim.lr_scheduler.CosineAnnealingLR(optimizer_top, T_max=EPOCHS_RETAIN)
        
        model_top.train()
        for epoch in range(EPOCHS_RETAIN):
            running_loss = 0.0
            for inputs, targets in loader_top:
                inputs, targets = inputs.to(DEVICE), targets.to(DEVICE)
                optimizer_top.zero_grad()
                loss = criterion(model_top(inputs), targets)
                loss.backward()
                optimizer_top.step()
                running_loss += loss.item()
            scheduler_top.step()
            
            if (epoch + 1) % 20 == 0 or epoch == 0:
                print(f"     [Top-{k}] Epoch {epoch+1}/{EPOCHS_RETAIN} | Loss: {running_loss/len(loader_top):.4f}")
                
        top_acc = evaluate_model(model_top, testloader)
        global_results["top_acc"][k].append(top_acc)
        print(f"  -> Top-{k} Model Accuracy: {top_acc:.2f}%")
        
        # Random-K
        print(f"  -> Training Random-{k} Removed Model...")
        random_remove = np.random.choice(all_indices, k, replace=False)
        indices_keep_rand = list(set(all_indices) - set(random_remove))
        subset_rand = Subset(trainset, indices_keep_rand)
        loader_rand = DataLoader(subset_rand, batch_size=BATCH_SIZE, shuffle=True, num_workers=8, pin_memory=True)
        
        model_rand = get_resnet18()
        optimizer_rand = optim.SGD(model_rand.parameters(), lr=LR, momentum=0.9, weight_decay=5e-4)
        scheduler_rand = optim.lr_scheduler.CosineAnnealingLR(optimizer_rand, T_max=EPOCHS_RETAIN)
        
        model_rand.train()
        for epoch in range(EPOCHS_RETAIN):
            running_loss = 0.0
            for inputs, targets in loader_rand:
                inputs, targets = inputs.to(DEVICE), targets.to(DEVICE)
                optimizer_rand.zero_grad()
                loss = criterion(model_rand(inputs), targets)
                loss.backward()
                optimizer_rand.step()
                running_loss += loss.item()
            scheduler_rand.step()
            
            if (epoch + 1) % 20 == 0 or epoch == 0:
                print(f"     [Rand-{k}] Epoch {epoch+1}/{EPOCHS_RETAIN} | Loss: {running_loss/len(loader_rand):.4f}")
                
        rand_acc = evaluate_model(model_rand, testloader)
        global_results["rand_acc"][k].append(rand_acc)
        print(f"  -> Random-{k} Model Accuracy: {rand_acc:.2f}%")

In [ ]:
# ==========================================
# 6. Class-Specific Experiment
# ==========================================
TARGET_CLASS = 3  
SEEDS = [42]  
CLASS_K_VALUES = [100, 250, 500, 1000] 

class_indices = [i for i, label in enumerate(trainset.targets) if label == TARGET_CLASS]
class_influence = [item for item in influence_ranking if item[0] in class_indices]
top_class_indices = [x[0] for x in class_influence]

class_results = {"k_values": CLASS_K_VALUES, "top_acc": {k: [] for k in CLASS_K_VALUES}, "rand_acc": {k: [] for k in CLASS_K_VALUES}}

print(f"\n--- Starting Class {TARGET_CLASS} Removal Experiment ---")
for k in CLASS_K_VALUES:
    print(f"\nEvaluating Class {TARGET_CLASS}, K = {k}")
    indices_keep_top = list(set(all_indices) - set(top_class_indices[:k]))
    subset_top = Subset(trainset, indices_keep_top)
    
    for seed in SEEDS: 
        set_seed(seed)
        
        # Class Top-K
        print(f"  -> Training Class {TARGET_CLASS} Top-{k} Removed Model...")
        loader_top = DataLoader(subset_top, batch_size=BATCH_SIZE, shuffle=True, num_workers=8, pin_memory=True)
        model_top = get_resnet18()
        optimizer_top = optim.SGD(model_top.parameters(), lr=LR, momentum=0.9, weight_decay=5e-4)
        scheduler_top = optim.lr_scheduler.CosineAnnealingLR(optimizer_top, T_max=EPOCHS_RETAIN)
        
        model_top.train()
        for epoch in range(EPOCHS_RETAIN):
            running_loss = 0.0
            for inputs, targets in loader_top:
                inputs, targets = inputs.to(DEVICE), targets.to(DEVICE)
                optimizer_top.zero_grad()
                loss = criterion(model_top(inputs), targets)
                loss.backward()
                optimizer_top.step()
                running_loss += loss.item()
            scheduler_top.step()
            
            if (epoch + 1) % 20 == 0 or epoch == 0:
                print(f"     [Class {TARGET_CLASS} Top-{k}] Epoch {epoch+1}/{EPOCHS_RETAIN} | Loss: {running_loss/len(loader_top):.4f}")
                
        top_acc = evaluate_model(model_top, testloader)
        class_results["top_acc"][k].append(top_acc)
        print(f"  -> Class {TARGET_CLASS} Top-{k} Model Accuracy: {top_acc:.2f}%")
        
        # Class Random-K
        print(f"  -> Training Class {TARGET_CLASS} Random-{k} Removed Model...")
        random_remove = np.random.choice(class_indices, k, replace=False)
        indices_keep_rand = list(set(all_indices) - set(random_remove))
        subset_rand = Subset(trainset, indices_keep_rand)
        loader_rand = DataLoader(subset_rand, batch_size=BATCH_SIZE, shuffle=True, num_workers=8, pin_memory=True)
        
        model_rand = get_resnet18()
        optimizer_rand = optim.SGD(model_rand.parameters(), lr=LR, momentum=0.9, weight_decay=5e-4)
        scheduler_rand = optim.lr_scheduler.CosineAnnealingLR(optimizer_rand, T_max=EPOCHS_RETAIN)
        
        model_rand.train()
        for epoch in range(EPOCHS_RETAIN):
            running_loss = 0.0
            for inputs, targets in loader_rand:
                inputs, targets = inputs.to(DEVICE), targets.to(DEVICE)
                optimizer_rand.zero_grad()
                loss = criterion(model_rand(inputs), targets)
                loss.backward()
                optimizer_rand.step()
                running_loss += loss.item()
            scheduler_rand.step()
            
            if (epoch + 1) % 20 == 0 or epoch == 0:
                print(f"     [Class {TARGET_CLASS} Rand-{k}] Epoch {epoch+1}/{EPOCHS_RETAIN} | Loss: {running_loss/len(loader_rand):.4f}")
                
        rand_acc = evaluate_model(model_rand, testloader)
        class_results["rand_acc"][k].append(rand_acc)
        print(f"  -> Class {TARGET_CLASS} Random-{k} Model Accuracy: {rand_acc:.2f}%")

In [21]:
# ==========================================
# 7. Visualization
# ==========================================
avg_top = [np.mean(global_results["top_acc"][k]) for k in K_VALUES]
avg_rand = [np.mean(global_results["rand_acc"][k]) for k in K_VALUES]
avg_class_top = [np.mean(class_results["top_acc"][k]) for k in CLASS_K_VALUES]
avg_class_rand = [np.mean(class_results["rand_acc"][k]) for k in CLASS_K_VALUES]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Plot 1: Global
ax1.axhline(y=base_acc, color='r', linestyle='--', label=f'Base Acc ({base_acc:.2f}%)')
ax1.plot(K_VALUES, avg_top, marker='o', color='b', label='Removed Top-K Influential')
ax1.plot(K_VALUES, avg_rand, marker='s', color='g', label='Removed Random-K')
ax1.set_title('Global Pruning: Impact on Accuracy')
ax1.set_xlabel('Samples Removed (K)')
ax1.set_ylabel('Test Accuracy (%)')
ax1.set_xticks(K_VALUES)
ax1.legend()
ax1.grid(True)

# Plot 2: Class-Specific
ax2.axhline(y=base_acc, color='r', linestyle='--', label=f'Base Acc ({base_acc:.2f}%)')
ax2.plot(CLASS_K_VALUES, avg_class_top, marker='o', color='b', label=f'Removed Top-K (Class {TARGET_CLASS})')
ax2.plot(CLASS_K_VALUES, avg_class_rand, marker='s', color='g', label=f'Removed Random-K (Class {TARGET_CLASS})')
ax2.set_title(f'Class {TARGET_CLASS} Pruning: Impact on Accuracy')
ax2.set_xlabel(f'Class {TARGET_CLASS} Samples Removed')
ax2.set_ylabel('Test Accuracy (%)')
ax2.set_xticks(CLASS_K_VALUES)
ax2.legend()
ax2.grid(True)

plt.tight_layout()
plt.show()

NameError: name 'K_VALUES' is not defined